In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pycaret

In [2]:
from pycaret.classification import *
from sklearn.model_selection import train_test_split

### 1. inicio de Pycaret
Se carga el conjunto de datos ya transformados (housing_limpio.csv) y se inicia el entorno de PyCaret con setup() indicando a target_clasificacion como la variable objetivo.
Esta función sera encarga automáticamente del dividir los datos en entrenamiento y los de prueba asi como de preparar el pipeline interno necesario para comparar múltiples modelos de clasificación de forma consistente.

In [3]:

df = pd.read_csv('../data/housing_limpio.csv')

print(df.shape)
display(df.head(3))

# Inicializar PyCaret con "target"
clf_setup = setup(
    data=df,
    target='target_clasificacion',
    session_id=302,
    html=False,
    verbose=True
)

(20640, 15)


,housing_median_age,median_income,target_clasificacion,rooms_per_household,bedrooms_per_room,population_per_household,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN,zona_california_Jefferson,zona_california_North California,zona_california_Silicon Valley,zona_california_South California,zona_california_West California
0,41.0,8.3252,1,6.984127,0.146591,2.555556,False,False,True,False,False,False,True,False,False
1,21.0,8.3014,1,6.238137,0.155797,2.109842,False,False,True,False,False,False,True,False,False
2,52.0,7.2574,1,8.288136,0.129516,2.802260,False,False,True,False,False,False,True,False,False


                    Description                 Value
0                    Session id                   302
1                        Target  target_clasificacion
2                   Target type                Binary
3           Original data shape           (20640, 15)
4        Transformed data shape           (20640, 15)
5   Transformed train set shape           (14447, 15)
6    Transformed test set shape            (6193, 15)
7              Numeric features                     5
8                    Preprocess                  True
9               Imputation type                simple
10           Numeric imputation                  mean
11       Categorical imputation                  mode
12               Fold Generator       StratifiedKFold
13                  Fold Number                    10
14                     CPU Jobs                    -1
15                      Use GPU                 False
16               Log Experiment                 False
17              Experiment N

In [4]:
#Comparar todos los modelos por el F1-Score
mejores_modelos = compare_models(sort='F1', n_select=3)

                                 Model  Accuracy     AUC  Recall   Prec.  \
rf            Random Forest Classifier    0.8770  0.9370  0.7523  0.8298   
gbc       Gradient Boosting Classifier    0.8758  0.9373  0.7482  0.8290   
et              Extra Trees Classifier    0.8725  0.9354  0.7381  0.8265   
ada               Ada Boost Classifier    0.8634  0.9255  0.7324  0.8037   
lr                 Logistic Regression    0.8612  0.9231  0.7328  0.7971   
knn             K Neighbors Classifier    0.8560  0.8974  0.7170  0.7927   
svm                SVM - Linear Kernel    0.8496  0.9190  0.6346  0.8436   
dt            Decision Tree Classifier    0.8211  0.7929  0.7202  0.7025   
lda       Linear Discriminant Analysis    0.8389  0.9000  0.6151  0.8126   
ridge                 Ridge Classifier    0.8386  0.9000  0.5909  0.8327   
nb                         Naive Bayes    0.6574  0.8510  0.9101  0.4693   
qda    Quadratic Discriminant Analysis    0.5896  0.6701  0.6235  0.4699   
dummy       

In [5]:
# Mostrar los 3 mejores modelos
display(mejores_modelos)

[RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                        criterion='gini', max_depth=None, max_features='sqrt',
                        max_leaf_nodes=None, max_samples=None,
                        min_impurity_decrease=0.0, min_samples_leaf=1,
                        min_samples_split=2, min_weight_fraction_leaf=0.0,
                        monotonic_cst=None, n_estimators=100, n_jobs=-1,
                        oob_score=False, random_state=302, verbose=0,
                        warm_start=False),
 GradientBoostingClassifier(ccp_alpha=0.0, criterion='friedman_mse', init=None,
                            learning_rate=0.1, loss='log_loss', max_depth=3,
                            max_features=None, max_leaf_nodes=None,
                            min_impurity_decrease=0.0, min_samples_leaf=1,
                            min_samples_split=2, min_weight_fraction_leaf=0.0,
                            n_estimators=100, n_iter_no_change=None,
    

In [6]:
from pycaret.classification import get_config

# Conjuntos internos generados por PyCaret
X_train_pycaret = get_config("X_train")
X_test_pycaret = get_config("X_test")
y_train_pycaret = get_config("y_train")
y_test_pycaret = get_config("y_test")

# Si también quieres los DataFrames completos (features + target):
df_train_pycaret = get_config("train")
df_test_pycaret = get_config("test")

In [7]:
X_train_pycaret.to_csv("../data/X_train_pycaret.csv", index=False)
X_test_pycaret.to_csv("../data/X_test_pycaret.csv", index=False)
y_train_pycaret.to_csv("../data/y_train_pycaret.csv", index=False)
y_test_pycaret.to_csv("../data/y_test_pycaret.csv", index=False)

In [8]:
# Guardar los mejores modelos seleccionados
save_model(mejores_modelos[0], '../models/RandomForest_Housing_Model')
save_model(mejores_modelos[1], '../models/GradientBoosting_Housing_Model')
save_model(mejores_modelos[2], '../models/ExtraTrees_Housing_Model')

Transformation Pipeline and Model Successfully Saved
Transformation Pipeline and Model Successfully Saved
Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['housing_median_age',
                                              'median_income',
                                              'rooms_per_household',
                                              'bedrooms_per_room',
                                              'population_per_household'],
                                     transformer=SimpleImputer(add_indicator=False,
                                                               copy=True,
                                                               fill_value=None,
                                                               keep_empty_features=False,
                                                               missing_values=nan,
                                                               strategy='mean'))),...
                  ExtraTreesCla

### 2. particion
La siguiente celda contiene una alternativa manual de partición entrenamiento/prueba usando train_test_split de scikit-learn con 85% para entrenamiento y estratificación por la variable objetivo que se deja comentada a modo de referencia.

Finalmente se opta por trabajar con los conjuntos de X_train, X_test, y_train y y_test generados por PyCaret en el paso anterior para mantener consistencia con el proceso de selección de modelos.

In [9]:
##separar (X) y la variable objetivo (y)
#X = df.drop(columns=['target_clasificacion'])
#y = df['target_clasificacion']

#Dividir test y traine (85%)
#X_train, X_test, y_train, y_test = train_test_split(
#    X, y,
#    test_size=0.15,
#    random_state=454,
#    stratify=y # Mantiene la proporción
#)

#unir train y test para exportar por separado
#train_data = pd.concat([X_train, y_train], axis=1)
#test_data = pd.concat([X_test, y_test], axis=1)

#train_data.to_csv('../data/housing_train.csv', index=False)
#test_data.to_csv('../data/housing_test.csv', index=False)


## 3. Conclucion:

Tras usar la librería de PyCaret para comparar automáticamente múltiples algoritmos de clasificación ordenados por el F1-score se seleccionaron los tres modelos con mejor desempeño que fueron: Random Forest, Gradient Boosting y Extra Trees, todos basados en los ensambles de árboles de decisión.
Estos tres modelos se guardaron para ser evaluados, comparados y optimizados en detalle en el siguiente notebook.